# WebCode2M — статистический анализ датасета

Ноутбук считает по датасету [`xcodemind/webcode2m_purified`](https://huggingface.co/datasets/xcodemind/webcode2m_purified)
метрики из `required_data.md`:

| # | Метрика | Как считается |
|---|---------|---------------|
| 1 | Количество примеров | точное `num_examples` из dataset info |
| 2 | Количество языков | нативное поле `lang` |
| 3 | Средний размер HTML | длина `text` в символах и байтах |
| 4 | Среднее количество DOM-узлов | lxml: элементы + текстовые узлы |
| 5 | Размер скриншота | разрешение в пикселях из поля `image` |
| 6 | Среднее количество CSS правил | все декларации `property: value` (инлайн + `<style>`) |
| 7 | Среднее количество выбранных источников | среднее число уникальных доменов на страницу |

Метрики 2–7 считаются по случайной выборке (`SAMPLE_SIZE`), метрика 1 — по всему
датасету. Тяжёлые объекты (изображения, HTML) в память не накапливаются: в цикле
берём только скалярные признаки.


## 1. Установка зависимостей

In [ ]:
%pip install -q datasets lxml pandas matplotlib tqdm

## 2. Импорты и конфигурация

In [ ]:
import os
import re
import logging
from urllib.parse import urlparse

import pandas as pd
import matplotlib.pyplot as plt
from lxml import html as lxml_html
from tqdm.auto import tqdm

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

import datasets
from datasets import load_dataset

logging.getLogger("httpx").setLevel(logging.WARNING)

DATASET = "xcodemind/webcode2m_purified"
SPLIT = "train"
SAMPLE_SIZE = 5000
# Случайная выборка из стрима вместо take() «с головы». shuffle() тасует порядок
# шардов (ключевой эффект: у этого датасета 2063 parquet-шарда, и без тасовки все
# SAMPLE_SIZE строк берутся из головы первого шарда → смещение) + локальный буфер.
# SEED фиксирует воспроизводимость. BUFFER держит столько недекодированных строк
# в RAM — снизить при нехватке памяти (тасовка шардов даёт разброс и при малом буфере).
SHUFFLE_SEED = 0
SHUFFLE_BUFFER = 10_000

print(f"datasets: {datasets.__version__}")
print(f"Датасет: {DATASET}, выборка: {SAMPLE_SIZE} (shuffle seed={SHUFFLE_SEED}, buffer={SHUFFLE_BUFFER})")

# --- Токенная длина кода (метрика 3, на замену символьной) ---
from token_len import count_tokens, recommend_max_length
from transformers import AutoTokenizer
# Точный id — когда SFT зафиксирует модель. Счётчики токенов у Qwen 2.5/3/3.5
# практически совпадают, поэтому это валидный прокси для оценки бюджета max_length.
TOKENIZER_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_TOKEN_BUDGET = 0   # бюджет визуальных токенов — заложить после согласования разрешения
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
print("tokenizer:", TOKENIZER_ID)


## 3. Метрика 1 — количество примеров

Значение получено суммированием `num_rows` из footer'ов всех 2063 parquet-шардов
сплита `train`. Пересчитывать при каждом запуске дорого, поэтому фиксируем
константой.

In [ ]:
TOTAL_EXAMPLES = 2_562_069
print(f"[Метрика 1] Количество примеров в датасете ({SPLIT}): {TOTAL_EXAMPLES:,}")

## 4. Загрузка стрим-датасета

In [ ]:
dataset = load_dataset(DATASET, streaming=True)
dataset

## 5. Функции-хелперы

Все метрики извлекаются из одного распарсенного DOM-дерева, чтобы парсить HTML
лишь один раз на пример.

In [ ]:
def count_dom_nodes(tree):
    # //text() включает whitespace-узлы (переносы строк между тегами) — фильтруем пустые,
    # иначе DOM-count завышается форматированием кода.
    return len(tree.xpath(".//*")) + len(tree.xpath("//text()[normalize-space()]"))


# Декларация property:value. Исключаем ложные срабатывания на url(http://...) и data:
# (лишние двоеточия в значениях) — берём первое двоеточие как разделитель property/value.
_DECL_RE = re.compile(r"[A-Za-z-]+\s*:[^;{}]+")

def _count_declarations(css_body):
    return sum(1 for _ in _DECL_RE.finditer(css_body))

def count_css_declarations(tree):
    total = 0
    for style_attr in tree.xpath("//*/@style"):
        total += _count_declarations(style_attr)
    for style_text in tree.xpath("//style//text()"):
        for block in re.findall(r"\{([^{}]*)\}", style_text):
            total += _count_declarations(block)
    return total


def count_unique_domains(tree):
    domains = set()
    for url in tree.xpath("//@src | //@href | //@data-src"):
        url = (url or "").strip()
        if not url or url.startswith(("#", "mailto:", "tel:", "javascript:", "data:")):
            continue
        parse_target = "http:" + url if url.startswith("//") else url
        netloc = urlparse(parse_target).netloc.lower()
        if netloc:
            domains.add(netloc)
    return len(domains)

## 6. Главный цикл по выборке

Итерируемся по `SAMPLE_SIZE` примерам, парсим HTML один раз и собираем только
скалярные признаки. Изображения и HTML в память не сохраняются.

In [ ]:
records = []

_sample = dataset[SPLIT].shuffle(seed=SHUFFLE_SEED, buffer_size=SHUFFLE_BUFFER).take(SAMPLE_SIZE)
for row in tqdm(_sample, total=SAMPLE_SIZE, desc="Обработка"):
    text = row.get("text") or ""
    img = row.get("image")
    img_w, img_h = (img.size if img is not None else (None, None))

    lang = row.get("lang")
    lang = str(lang).strip().lower() if lang else "unknown"

    rec = {
        "html_chars": len(text),
        "html_tokens": count_tokens(text, tokenizer),
        "html_bytes": len(text.encode("utf-8")),
        "img_w": img_w,
        "img_h": img_h,
        "dom_nodes": None,
        "css_decls": None,
        "n_domains": None,
        "lang": lang,
        "parse_ok": False,
    }
    try:
        tree = lxml_html.fromstring(text)
        rec["dom_nodes"] = count_dom_nodes(tree)
        rec["css_decls"] = count_css_declarations(tree)
        rec["n_domains"] = count_unique_domains(tree)
        rec["parse_ok"] = True
    except Exception:
        pass

    records.append(rec)

print(f"Собрано записей: {len(records)}, распарсено успешно: {sum(r['parse_ok'] for r in records)}")

## 7. DataFrame с признаками

In [ ]:
df = pd.DataFrame(records)
print(f"Размер списка признаков в памяти: {df.memory_usage(deep=True).sum() / 1024**2:.2f} МБ")
df.head()

## 8. Агрегированные метрики

Считаем только по успешно распарсенным примерам.

In [ ]:
ok = df[df["parse_ok"]]

agg = pd.DataFrame({
    "mean":   [ok["dom_nodes"].mean(), ok["css_decls"].mean(), ok["n_domains"].mean(), ok["img_h"].mean()],
    "median": [ok["dom_nodes"].median(), ok["css_decls"].median(), ok["n_domains"].median(), ok["img_h"].median()],
    "std":    [ok["dom_nodes"].std(), ok["css_decls"].std(), ok["n_domains"].std(), ok["img_h"].std()],
    "min":    [ok["dom_nodes"].min(), ok["css_decls"].min(), ok["n_domains"].min(), ok["img_h"].min()],
    "max":    [ok["dom_nodes"].max(), ok["css_decls"].max(), ok["n_domains"].max(), ok["img_h"].max()],
}, index=["DOM-узлы", "CSS декларации", "Уникальные домены", "Высота скриншота, px"])

agg.round(1)

### 8b. Среднее vs медиана — проверка на скошенность

По гистограммам похоже, что размеры (HTML/DOM/CSS) сильно скошены вправо:
небольшое число очень крупных страниц тянет среднее заметно выше медианы.
Ниже — прямое сравнение `mean` и `median` по всем размерным метрикам плюс
коэффициент скошенности `mean / median` (чем сильнее он выше 1, тем длиннее
правый хвост распределения) и `p90`/`p99` для масштаба хвоста.

In [ ]:
_skew_cols = {
    "Код (токены)": "html_tokens",
    "DOM-узлы": "dom_nodes",
    "CSS декларации": "css_decls",
    "Уникальные домены": "n_domains",
    "Высота скриншота, px": "img_h",
}

skew = pd.DataFrame({
    "median": [ok[c].median() for c in _skew_cols.values()],
    "mean":   [ok[c].mean() for c in _skew_cols.values()],
    "p90":    [ok[c].quantile(0.90) for c in _skew_cols.values()],
    "p99":    [ok[c].quantile(0.99) for c in _skew_cols.values()],
    "max":    [ok[c].max() for c in _skew_cols.values()],
}, index=list(_skew_cols.keys()))

skew["mean / median"] = (skew["mean"] / skew["median"]).round(2)
skew = skew[["median", "mean", "mean / median", "p90", "p99", "max"]]
skew.round(1)

## 9. Метрика 2 — количество языков

Источник — нативное поле `lang` (языковая разметка авторов датасета: именно на
этих языках учится модель).

In [ ]:
lang_counts = df["lang"].value_counts()
n_langs = df.loc[df["lang"] != "unknown", "lang"].nunique()

print(f"[Метрика 2] Уникальных языков (поле lang): {n_langs}")
print("\nТоп-15 языков:")
lang_counts.head(15)

In [ ]:
top_langs = lang_counts.head(15)
plt.figure(figsize=(9, 4))
top_langs.plot(kind="bar")
plt.title("Распределение языков (поле lang, топ-15)")
plt.ylabel("Кол-во примеров")
plt.xlabel("Код языка")
plt.tight_layout()
plt.show()

## 10. Гистограммы распределений

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

ok["html_tokens"].hist(ax=axes[0, 0], bins=40)
axes[0, 0].set_title("Размер кода (токены)")

ok["dom_nodes"].hist(ax=axes[0, 1], bins=40)
axes[0, 1].set_title("Кол-во DOM-узлов")

ok["css_decls"].hist(ax=axes[0, 2], bins=40)
axes[0, 2].set_title("Кол-во CSS-деклараций")

ok["n_domains"].hist(ax=axes[1, 0], bins=40)
axes[1, 0].set_title("Уникальных доменов на страницу")

ok["img_h"].hist(ax=axes[1, 1], bins=40)
axes[1, 1].set_title("Высота скриншота, px")

ok["img_w"].hist(ax=axes[1, 2], bins=40)
axes[1, 2].set_title("Ширина скриншота, px")

plt.tight_layout()
plt.show()

## 11. Итоговая сводка

Сопоставление с каждым пунктом `required_data.md`.

In [ ]:
def fmt(x):
    return f"{x:,.1f}" if x is not None else "n/a"

w_mode = ok["img_w"].mode()
w_mode = int(w_mode.iloc[0]) if len(w_mode) else None

summary = pd.DataFrame([
    ("1. Количество примеров",                 f"{TOTAL_EXAMPLES:,}" if TOTAL_EXAMPLES else "n/a"),
    ("2. Количество языков",                   f"{n_langs}"),
    ("3. Средний размер кода (токены, Qwen)",  fmt(ok["html_tokens"].mean())),
    ("   p99 токенов (ориентир для max_length)", fmt(ok["html_tokens"].quantile(0.99))),
    ("4. Среднее кол-во DOM-узлов",            fmt(ok["dom_nodes"].mean())),
    ("5. Размер скриншота (W x H, px)",        f"{w_mode} x {fmt(ok['img_h'].mean())} (средняя высота)"),
    ("6. Среднее кол-во CSS правил",           fmt(ok["css_decls"].mean())),
    ("7. Среднее кол-во уникальных доменов",   fmt(ok["n_domains"].mean())),
], columns=["Метрика (required_data.md)", "Значение"])

print(f"Выборка для метрик 2-7: {len(ok):,} успешно распарсенных примеров "
      f"(из {len(df):,} загруженных)\n")
summary

## Рекомендуемый max_length (токены)

In [ ]:
# Ориентир max_length (контракт SFT): p99 длины кода в токенах, округл. вверх до 64,
# + бюджет визуальных токенов. Та же логика, что compute_max_length в token_len.py.
_tok_col = "code_tokens" if "code_tokens" in ok.columns else "html_tokens"
_rec = recommend_max_length(ok[_tok_col].tolist(), quantile=0.99, round_to=64,
                            image_token_budget=IMAGE_TOKEN_BUDGET)
print(f"Токены кода: median={ok[_tok_col].median():.0f}, "
      f"p99={ok[_tok_col].quantile(0.99):.0f}, max={ok[_tok_col].max():.0f}")
print(f"Рекомендуемый max_length (p99 → округл. 64, +img {IMAGE_TOKEN_BUDGET}): {_rec}")

## 12. Все графики вместе

Те же графики из разделов 9–10, собранные в одну сетку — для скриншота.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

top_langs.plot(kind="bar", ax=axes[0])
axes[0].set_title("Языки (топ-15)")
axes[0].set_ylabel("Кол-во примеров")

ok["html_tokens"].hist(ax=axes[1], bins=40)
axes[1].set_title("Размер кода (токены)")

ok["dom_nodes"].hist(ax=axes[2], bins=40)
axes[2].set_title("Кол-во DOM-узлов")

ok["css_decls"].hist(ax=axes[3], bins=40)
axes[3].set_title("Кол-во CSS-деклараций")

ok["n_domains"].hist(ax=axes[4], bins=40)
axes[4].set_title("Уникальных доменов на страницу")

ok["img_h"].hist(ax=axes[5], bins=40)
axes[5].set_title("Высота скриншота, px")

ok["img_w"].hist(ax=axes[6], bins=40)
axes[6].set_title("Ширина скриншота, px")

axes[7].axis("off")

fig.suptitle("WebCode2M — все графики", fontsize=14)
plt.tight_layout()
plt.show()